In [2]:
# Naive Bayes with dataset from URL

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# Load dataset (public CSV URL)
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
data = pd.read_csv(url)

# Encode target (species)
le = LabelEncoder()
data['species'] = le.fit_transform(data['species'])

# Features and target
X = data.drop('species', axis=1)
y = data['species']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = GaussianNB()

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Accuracy
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred))

Naive Bayes Accuracy: 1.0


In [ ]:
pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 45.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
!pip install pgmpy
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from sklearn.preprocessing import LabelEncoder

# Load dataset
data = pd.read_csv("F1(Drivers)-Dataset-E0124099.csv")

print("Columns:", data.columns)

# --------- PREPROCESSING ---------

# Drop unnecessary columns
drop_cols = [col for col in data.columns if "id" in col.lower() or "Unnamed" in col.lower()]
data = data.drop(columns=drop_cols, errors='ignore')

# Rename columns for consistency if needed, or identify target/features based on original names
# For this dataset, 'Championships' seems like a good target.

# Convert all columns to categorical
le = LabelEncoder()
for col in data.columns:
    # Only encode if the column is not already numeric (e.g., 'Seasons', 'Championships')
    # or if it contains string/object types.
    if data[col].dtype == 'object' or data[col].nunique() < len(data[col]) * 0.5: # Heuristic for categorical
        data[col] = le.fit_transform(data[col].astype(str))

# --------- TARGET COLUMN ---------
# Based on the F1 dataset, 'Championships' seems like a suitable target.
# If 'Championships' is not present or you have a different target in mind, please specify.
if 'Championships' in data.columns:
    target_col = 'Championships'
else:
    raise Exception("Target column 'Championships' not found! Please choose an existing column.")

print("Target:", target_col)

# --------- FEATURES ---------
# Selecting relevant features for an F1 driver dataset. Adjust as needed.
possible_features = ['Nationality', 'Seasons', 'Race_Entries', 'Race_Starts', 'Poles', 'Wins', 'Fastest_Laps', 'Podiums']
features = [f for f in possible_features if f in data.columns and f != target_col]

if not features:
    raise Exception("No valid features found after filtering. Please check your dataset columns and 'possible_features' list.")

print("Features:", features)

# --------- MODEL ---------
edges = [(f, target_col) for f in features]

# Adding a check for unconnected nodes in the model (optional but good practice)
# This is more for complex networks, but ensures all features are connected.

model = DiscreteBayesianNetwork(edges)

# Train model
# Ensure data passed to fit has both features and target
model.fit(data[features + [target_col]], estimator=MaximumLikelihoodEstimator)

# --------- INFERENCE ---------
infer = VariableElimination(model)

# Example evidence (safe selection)
evidence = {}
# Select a few features to use as evidence, ensuring they exist in the data and model
if 'Seasons' in features:
    evidence['Seasons'] = data['Seasons'].iloc[0]
if 'Race_Entries' in features:
    evidence['Race_Entries'] = data['Race_Entries'].iloc[0]

# If no evidence is set, the query might still run but with marginal probabilities
if not evidence and features:
    print("Warning: No specific evidence set. Querying marginal probability for target.")

# Query
# Ensure the variable to query (target_col) is in the model
result = infer.query(variables=[target_col], evidence=evidence)

print("\nDiagnosis Result:")
print(result)


In [ ]:
# Neural Network with proper preprocessing

import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset (public URL)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

# Column names
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
           'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

data = pd.read_csv(url, names=columns)

# Features and target
X = data.drop('Outcome', axis=1)
y = data['Outcome']

# Scaling (VERY IMPORTANT)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(X_train, y_train, epochs=20, batch_size=16, verbose=1)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("Neural Network Accuracy:", acc)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6824 - loss: 0.6009
Epoch 2/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7296 - loss: 0.5200
Epoch 3/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7622 - loss: 0.4896
Epoch 4/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7769 - loss: 0.4638
Epoch 5/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7736 - loss: 0.4638
Epoch 6/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7769 - loss: 0.4468
Epoch 7/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7769 - loss: 0.4554
Epoch 8/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7785 - loss: 0.4435
Epoch 9/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7866 - loss: 0.4433
Epoch 10/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7801 - loss: 0.4436
Epoch 11/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7866 - loss: 0.4486
Epoch 12/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7818 - loss: 0.4426
